# OpenMM Workflow

OpenMM is a high-performance toolkit for molecular simulation. MolSysMT provides two-way interoperability with OpenMM, allowing users to build and parameterize systems, run energy minimizations and MD simulations, and seamlessly retrieve coordinates and thermodynamic properties for downstream analysis.


## Building the Molecular System

We start by constructing an alanine dipeptide (`AceAlaNme`) with MolSysMT:


In [1]:
import molsysmt as msm
from molsysmt import pyunitwizard as puw

molsys = msm.build.build_peptide('AceAlaNme')
msm.info(molsys)


form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.MolSys,22,3,1,1,1,1,1,1


## Interoperability and Conversions

MolSysMT can convert native systems directly into OpenMM objects, including `openmm.Topology` and fully configured `openmm.app.Simulation` instances:


In [2]:
omm_topology = msm.convert(molsys, to_form='openmm.Topology')
print(omm_topology)


<Topology; 1 chains, 3 residues, 22 atoms, 21 bonds>


In [3]:
simulation = msm.convert(molsys, to_form='openmm.Simulation', forcefield='AMBER14')
print(simulation)


## Energy Minimization

We query the potential energy of the initial conformation and minimize it with OpenMM:


In [4]:
e_initial = msm.molecular_mechanics.get_potential_energy(simulation)
print(f"Initial potential energy: {e_initial}")

simulation.minimizeEnergy(maxIterations=100)

e_min = msm.molecular_mechanics.get_potential_energy(simulation)
print(f"Minimized potential energy: {e_min}")


Initial potential energy: -55.7942228620533 kilojoule / mole
Minimized potential energy: -86.12218322364657 kilojoule / mole


## Molecular Dynamics Simulation

We can integrate equations of motion directly through the OpenMM simulation object:


In [5]:
simulation.step(500)
e_final = msm.molecular_mechanics.get_potential_energy(simulation)
print(f"Potential energy after 500 MD steps: {e_final}")


Potential energy after 500 MD steps: -86.87117403639718 kilojoule / mole


## Extracting Simulation States

We retrieve the updated coordinates from OpenMM and store them back into our MolSysMT system:


In [6]:
coords_simulated = msm.get(simulation.context, coordinates=True)
msm.set(molsys, coordinates=coords_simulated)
print(f"Updated MolSys coordinates shape: {msm.get(molsys, coordinates=True).shape}")


Updated MolSys coordinates shape: (1, 22, 3)


## Interactive Visualization

We visualize the simulated conformation in 3D using MolSysViewer:


In [7]:
msm.view(molsys)


'<iframe src="../../_static/views/showcase_openmm.html" width="100%" height="480px" style="border:none;"></iframe>'